In [ ]:
import IPython
import numpy as np
import tvm
from tvm.ir.module import IRModule
from tvm.script import tir as T

然后，让我们尝试做一些具有挑战性的事情：二维卷积。这是图像处理中的常见操作。

这是使用 NCHW 布局的卷积的数学定义：

<math xmlns="http://www.w3.org/1998/Math/MathML" display="block">
  <mi>C</mi>
  <mi>o</mi>
  <mi>n</mi>
  <mi>v</mi>
  <mo stretchy="false">[</mo>
  <mi>b</mi>
  <mo>,</mo>
  <mi>k</mi>
  <mo>,</mo>
  <mi>i</mi>
  <mo>,</mo>
  <mi>j</mi>
  <mo stretchy="false">]</mo>
  <mo>=</mo>
  <munder>
    <mo data-mjx-texclass="OP">&#x2211;</mo>
    <mrow data-mjx-texclass="ORD">
      <mi>d</mi>
      <mi>i</mi>
      <mo>,</mo>
      <mi>d</mi>
      <mi>j</mi>
      <mo>,</mo>
      <mi>q</mi>
    </mrow>
  </munder>
  <mi>A</mi>
  <mo stretchy="false">[</mo>
  <mi>b</mi>
  <mo>,</mo>
  <mi>q</mi>
  <mo>,</mo>
  <mi>s</mi>
  <mi>t</mi>
  <mi>r</mi>
  <mi>i</mi>
  <mi>d</mi>
  <mi>e</mi>
  <mi>s</mi>
  <mo>&#x2217;</mo>
  <mi>i</mi>
  <mo>+</mo>
  <mi>d</mi>
  <mi>i</mi>
  <mo>,</mo>
  <mi>s</mi>
  <mi>t</mi>
  <mi>r</mi>
  <mi>i</mi>
  <mi>d</mi>
  <mi>e</mi>
  <mi>s</mi>
  <mo>&#x2217;</mo>
  <mi>j</mi>
  <mo>+</mo>
  <mi>d</mi>
  <mi>j</mi>
  <mo stretchy="false">]</mo>
  <mo>&#x2217;</mo>
  <mi>W</mi>
  <mo stretchy="false">[</mo>
  <mi>k</mi>
  <mo>,</mo>
  <mi>q</mi>
  <mo>,</mo>
  <mi>d</mi>
  <mi>i</mi>
  <mo>,</mo>
  <mi>d</mi>
  <mi>j</mi>
  <mo stretchy="false">]</mo>
  <mo>,</mo>
</math>
 
其中，A 是输入张量，W 是权重张量，b 是批次索引，k 是输出通道，i 和 j 是图像高度和宽度的索引，di 和 dj 是权重的索引，q 是输入通道，strides 是过滤器窗口的步幅。

在练习中，我们选择了一个小而简单的情况，即 stride=1, padding=0。

In [33]:
N, CI, H, W, CO, K = 1, 1, 8, 8, 2, 3
OUT_H, OUT_W = H - K + 1, W - K + 1
data = np.arange(N*CI*H*W).reshape(N, CI, H, W)
weight = np.arange(CO*CI*K*K).reshape(CO, CI, K, K)
data, weight

(array([[[[ 0,  1,  2,  3,  4,  5,  6,  7],
          [ 8,  9, 10, 11, 12, 13, 14, 15],
          [16, 17, 18, 19, 20, 21, 22, 23],
          [24, 25, 26, 27, 28, 29, 30, 31],
          [32, 33, 34, 35, 36, 37, 38, 39],
          [40, 41, 42, 43, 44, 45, 46, 47],
          [48, 49, 50, 51, 52, 53, 54, 55],
          [56, 57, 58, 59, 60, 61, 62, 63]]]]),
 array([[[[ 0,  1,  2],
          [ 3,  4,  5],
          [ 6,  7,  8]]],
 
 
        [[[ 9, 10, 11],
          [12, 13, 14],
          [15, 16, 17]]]]))

In [40]:
# torch version
import torch

data_torch = torch.Tensor(data)
weight_torch = torch.Tensor(weight)
conv_torch = torch.nn.functional.conv2d(data_torch, weight_torch)
conv_torch = conv_torch.numpy().astype(np.int64)
conv_torch

array([[[[ 474,  510,  546,  582,  618,  654],
         [ 762,  798,  834,  870,  906,  942],
         [1050, 1086, 1122, 1158, 1194, 1230],
         [1338, 1374, 1410, 1446, 1482, 1518],
         [1626, 1662, 1698, 1734, 1770, 1806],
         [1914, 1950, 1986, 2022, 2058, 2094]],

        [[1203, 1320, 1437, 1554, 1671, 1788],
         [2139, 2256, 2373, 2490, 2607, 2724],
         [3075, 3192, 3309, 3426, 3543, 3660],
         [4011, 4128, 4245, 4362, 4479, 4596],
         [4947, 5064, 5181, 5298, 5415, 5532],
         [5883, 6000, 6117, 6234, 6351, 6468]]]])

In [32]:
data = np.arange(9).reshape(1, 1, 3, 3)
weight = np.arange(18).reshape(2, 1, 3, 3)
data_torch = torch.Tensor(data)
weight_torch = torch.Tensor(weight)
conv_torch = torch.nn.functional.conv2d(data_torch, weight_torch)
conv_torch = conv_torch.numpy().astype(np.int64)
conv_torch

array([[[[204]],

        [[528]]]])

In [51]:
@tvm.script.ir_module
class MyConv2d:
    @T.prim_func
    def Conv2d( A:T.Buffer((1, 1, 8, 8), "int64"),
                B:T.Buffer((2, 1, 3, 3), "int64"),
                C:T.Buffer((1, 2, 6, 6), "int64"),
                ):
        T.func_attr({"global_symbol":"Conv2d", "tir.noalias":True})
        # for b, k, i ,j in T.grid(1, 2, 6, 6):
        #     with T.block("Conv"):
        #         vb, vk, vi, vj = T.axis.remap("SSSS", [b, k, i ,j])
        #         with T.init():
        #             C[vb, vk, vi, vj] = T.int64(0)
        #         for q, di, dj in T.grid(1, 3, 3):
        #             with T.block("Matmul"):
        #                 vq, vdi, vdj = T.axis.remap("SSS", [q, di ,dj])
        #                 C[vb, vk, vi, vj] = A[vb, vq, vi+vdi, vj+vdj] * B[vk, vq, vdi, vdj] + C[vb, vk, vi, vj]
        for b, q, k, i , j, di, dj in T.grid(1, 1, 2, 6, 6, 3, 3):
            with T.block("Conv"):
                vb, vq, vk, vi, vj, vdi, vdj = T.axis.remap("SSSSSRR", [b, q, k, i, j, di, dj])
                with T.init():
                    C[vb, vk, vi, vj] = T.int64(0)
                C[vb, vk, vi, vj] = A[vb, vq, vi+vdi, vj+vdj] * B[vk, vq, vdi, vdj] + C[vb, vk, vi, vj]


In [52]:
rt_lib = tvm.build(MyConv2d, target="llvm")
data_tvm = tvm.nd.array(data)
weight_tvm = tvm.nd.array(weight)
conv_tvm = tvm.nd.array(np.empty((N, CO, OUT_H, OUT_W), dtype=np.int64))
rt_lib["Conv2d"](data_tvm, weight_tvm, conv_tvm)
conv_tvm,conv_torch
np.testing.assert_allclose(conv_tvm.numpy(), conv_torch, rtol=1e-5)

这里可能产生的问题是 有关T.init()对C初始化的问题 如果di dj不设置为规约的话 会导致在di dj两层循环中C每次都被置零无法记录之前保存的和